# Part 2: Integration of Environmental Covariates & Ablation Study

Ce notebook réalise la deuxième partie du projet :
1. **Data Preparation** : Traitement des CSV exportés depuis GEE (contenant Sentinel-2 + Climat + Sol + Topographie) vers un format `.npz` utilisable par PyTorch.
2. **Exploratory Analysis (EDA)** : Génération et affichage de statistiques et de matrices de corrélation pour comprendre l'impact des covariables sur les classes de cultures.
3. **Ablation Study** : Entraînement de l'architecture MCTNet sous 5 configurations différentes (Baseline, Climate, Soil, Topo, All) afin d'évaluer l'apport de chaque groupe de variables environnementales.

In [ ]:
import sys
import json
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image

try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive monté avec succès.')
except ImportError:
    print('Google Colab non détecté. Exécution en local.')

# --- CONFIGURATION DES CHEMINS ---
# À adapter selon l'arborescence de votre Google Drive
PROJECT_DIR = Path('/content/drive/MyDrive/New project/python')  # Dossier contenant vos scripts .py
DRIVE_BASE_DIR = Path('/content/drive/MyDrive/mctnet_env_covariates_2021') # Dossier des exports GEE

PROCESSED_DIR = DRIVE_BASE_DIR / 'processed_env'
EDA_DIR = DRIVE_BASE_DIR / 'eda_results'
ABLATION_DIR = DRIVE_BASE_DIR / 'ablation_study_runs'

sys.path.insert(0, str(PROJECT_DIR))

# Fichiers CSV attendus (Exportés via mctnet_env_covariates_prep_2021.js)
CSV_AR = DRIVE_BASE_DIR / 'mctnet_env_samples_AR_2021.csv'
CSV_CA = DRIVE_BASE_DIR / 'mctnet_env_samples_CA_2021.csv'

print(f"CSV Arkansas trouvé : {CSV_AR.exists()}")
print(f"CSV California trouvé : {CSV_CA.exists()}")

## 1. Data Preparation
Nous utilisons le script `build_mctnet_env_dataset.py` pour convertir les données CSV brutes en tenseurs PyTorch. Les variables environnementales sont extraites séparément de la série temporelle.

In [ ]:
import subprocess

print("Lancement de la préparation des données avec covariables...")

cmd_prep = [
    "python", str(PROJECT_DIR / "build_mctnet_env_dataset.py"),
    "--input-csv", str(CSV_AR), str(CSV_CA),
    "--output-dir", str(PROCESSED_DIR)
]

result_prep = subprocess.run(cmd_prep, capture_output=True, text=True)
print(result_prep.stdout)
if result_prep.stderr:
    print("Erreurs :", result_prep.stderr)

## 2. Exploratory Data Analysis (EDA)
Nous lançons le script `environmental_covariate_eda.py` pour analyser la corrélation entre les variables environnementales et les types de cultures.

In [ ]:
print("Lancement de l'analyse exploratoire (EDA) pour l'Arkansas...")
cmd_eda_ar = [
    "python", str(PROJECT_DIR / "environmental_covariate_eda.py"),
    "--input-csv", str(CSV_AR),
    "--output-dir", str(EDA_DIR / "Arkansas")
]
subprocess.run(cmd_eda_ar, check=True)

print("Lancement de l'analyse exploratoire (EDA) pour la Californie...")
cmd_eda_ca = [
    "python", str(PROJECT_DIR / "environmental_covariate_eda.py"),
    "--input-csv", str(CSV_CA),
    "--output-dir", str(EDA_DIR / "California")
]
subprocess.run(cmd_eda_ca, check=True)
print("EDA terminée avec succès.")

In [ ]:
# Affichage des résultats de l'EDA pour l'Arkansas
print("--- Matrice de Corrélation de Pearson (Arkansas) ---")
display(Image(filename=str(EDA_DIR / 'Arkansas' / 'covariate_pearson_heatmap.png')))

print("\n--- Corrélation (Eta) entre les classes de cultures et les covariables (Arkansas) ---")
display(Image(filename=str(EDA_DIR / 'Arkansas' / 'covariate_class_eta_barplot.png')))

print("\n--- Valeurs moyennes des covariables par classe de culture (Arkansas) ---")
display(Image(filename=str(EDA_DIR / 'Arkansas' / 'covariate_class_means_heatmap.png')))

## 3. Ablation Study
L'objectif est d'entraîner le modèle avec différentes combinaisons de données. 
**Stratégie de fusion spatio-temporelle (Early Fusion)** :
Les données environnementales sont statiques. Pour les intégrer au MCTNet sans modifier son architecture interne, nous allons les concaténer aux 10 bandes Sentinel-2 pour chaque pas de temps. Ainsi, si on ajoute le Climat (6 variables), l'`input_dim` passera de 10 à 16.

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

from mctnet_model import build_mctnet
from train_mctnet import run_epoch, set_seed

class CropAblationDataset(Dataset):
    def __init__(self, bundle, split, env_indices):
        self.x = torch.from_numpy(bundle[f'x_{split}']).float()
        self.valid_mask = torch.from_numpy(bundle[f'valid_mask_{split}']).float()
        self.y = torch.from_numpy(bundle[f'y_{split}']).long()
        self.env = torch.from_numpy(bundle[f'env_{split}']).float()
        self.env_indices = env_indices

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        x_seq = self.x[idx] # Shape: [36, 10]
        
        if self.env_indices:
            # Extraction des variables choisies (Climate, Soil, etc.)
            env_vals = self.env[idx][self.env_indices] # Shape: [num_env_vars]
            # On répète les variables statiques sur les 36 pas de temps
            env_seq = env_vals.unsqueeze(0).repeat(x_seq.shape[0], 1) # Shape: [36, num_env_vars]
            # Concaténation temporelle: Sentinel-2 + Covariables
            x_seq = torch.cat([x_seq, env_seq], dim=1) # Shape: [36, 10 + num_env_vars]

        return {'x': x_seq, 'valid_mask': self.valid_mask[idx], 'y': self.y[idx]}

def get_env_indices(metadata, config_name):
    if config_name == 'baseline':
        return []
    if config_name == 'all':
        return list(range(len(metadata['environmental_covariates']['all_columns'])))
    return metadata['environmental_covariates']['group_to_indices'][config_name]

In [ ]:
def train_ablation_config(state_slug, config_name, epochs=50, batch_size=32, lr=1e-3):
    npz_path = PROCESSED_DIR / f'{state_slug}_mctnet_env_dataset.npz'
    json_path = PROCESSED_DIR / f'{state_slug}_mctnet_env_dataset.json'
    
    bundle = np.load(npz_path, allow_pickle=True)
    bundle_dict = {key: bundle[key] for key in bundle.files}
    metadata = json.loads(json_path.read_text(encoding='utf-8'))
    
    env_indices = get_env_indices(metadata, config_name)
    input_dim = 10 + len(env_indices)
    num_classes = len(metadata['class_name_to_index'])
    
    loaders = {}
    for split in ['train', 'val', 'test']:
        dataset = CropAblationDataset(bundle_dict, split, env_indices)
        loaders[split] = DataLoader(dataset, batch_size=batch_size, shuffle=(split=='train'))
        
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = build_mctnet(num_classes=num_classes, input_dim=input_dim).to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    
    best_val_kappa = -1.0
    best_test_metrics = {}
    
    print(f"\n--- Training [{state_slug.upper()}] Config: {config_name.upper()} (Input Dim: {input_dim}) ---")
    
    for epoch in range(1, epochs + 1):
        train_metrics = run_epoch(model, loaders['train'], criterion, device, optimizer)
        val_metrics = run_epoch(model, loaders['val'], criterion, device, None)
        test_metrics = run_epoch(model, loaders['test'], criterion, device, None)
        
        if val_metrics['kappa'] > best_val_kappa:
            best_val_kappa = val_metrics['kappa']
            best_test_metrics = test_metrics
            
        if epoch % 10 == 0 or epoch == epochs:
            print(f"Epoch {epoch:02d}/{epochs} | Val OA: {val_metrics['oa']:.4f} | Val Kappa: {val_metrics['kappa']:.4f} | Test F1: {test_metrics['macro_f1']:.4f}")
            
    print(f"=> Meilleures Test Metrics pour {config_name}: OA={best_test_metrics['oa']:.4f}, Kappa={best_test_metrics['kappa']:.4f}, F1={best_test_metrics['macro_f1']:.4f}")
    return best_test_metrics

In [ ]:
# --- EXÉCUTION DE L'ABLATION STUDY ---
set_seed(2021)

configurations = ['baseline', 'climate', 'soil', 'topography', 'all']
states = ['arkansas', 'california']

results = {state: {} for state in states}

for state in states:
    for config in configurations:
        # On réduit les epochs à 50 pour l'étude d'ablation pour accélérer, 
        # ajustez à 100-200 pour le rapport final.
        res = train_ablation_config(state, config, epochs=50)
        results[state][config] = res

# Sauvegarde des résultats de l'ablation
ABLATION_DIR.mkdir(parents=True, exist_ok=True)
with open(ABLATION_DIR / 'ablation_results.json', 'w') as f:
    json.dump(results, f, indent=2)

## 4. Analyse et Visualisation des Résultats
Nous affichons un graphique comparatif pour visualiser l'apport de chaque variable environnementale par rapport à la baseline (Sentinel-2 seul).

In [ ]:
def plot_ablation_results(results_dict, metric='oa', title='Overall Accuracy (OA) Comparison'):
    configs = list(results_dict['arkansas'].keys())
    
    ar_vals = [results_dict['arkansas'][c][metric] for c in configs]
    ca_vals = [results_dict['california'][c][metric] for c in configs]

    x = np.arange(len(configs))
    width = 0.35

    fig, ax = plt.subplots(figsize=(10, 6))
    rects1 = ax.bar(x - width/2, ar_vals, width, label='Arkansas', color='#2563eb')
    rects2 = ax.bar(x + width/2, ca_vals, width, label='California', color='#16a34a')

    ax.set_ylabel(metric.upper())
    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels([c.capitalize() for c in configs])
    ax.legend()
    ax.set_ylim(0.7, 1.0) # Ajusté pour la lisibilité
    ax.grid(axis='y', linestyle='--', alpha=0.7)

    fig.tight_layout()
    plt.savefig(ABLATION_DIR / f'ablation_comparison_{metric}.png', dpi=200)
    plt.show()

print("Visualisation des performances (OA) :")
plot_ablation_results(results, metric='oa', title='Ablation Study: Overall Accuracy (OA)')

print("Visualisation des performances (Macro F1) :")
plot_ablation_results(results, metric='macro_f1', title='Ablation Study: Macro F1 Score')